# 01 — Diagnóstico del archivo Bronze de telemetría

## ¿Qué haremos en este notebook?

Revisaremos el archivo original de sensores de AndinaLog para conocer su estructura y detectar posibles problemas antes de limpiar los datos.

**Importante:** este notebook solamente diagnostica. No corrige, elimina ni reemplaza valores.

La **unidad de observación esperada** es una lectura de telemetría de un viaje en un momento determinado.

## 1. Preparación y carga del archivo

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

# Ruta local proporcionada para esta tarea.
# En Google Colab, cambia esta ruta por la ubicación del CSV en Drive o /content.
RUTA_CSV = Path(
    r"C:/Users/remrodri/Github/practicasNotebookColab/"
    r"proyecto-integradorV2/datos/bronze/andinalog_iot_telemetry.csv"
)

if not RUTA_CSV.exists():
    alternativa = Path("/content/andinalog_iot_telemetry.csv")
    if alternativa.exists():
        RUTA_CSV = alternativa
    else:
        raise FileNotFoundError(
            "No se encontró el CSV. Ajusta RUTA_CSV o súbelo a "
            "/content/andinalog_iot_telemetry.csv en Colab."
        )

# Se lee como texto para observar los valores originales sin modificarlos.
bronze = pd.read_csv(RUTA_CSV, dtype=str, keep_default_na=False)

print("Archivo cargado:", RUTA_CSV)
print(f"Tamaño: {bronze.shape[0]:,} filas y {bronze.shape[1]} columnas")
print(sns.__version__)

Archivo cargado: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2\datos\bronze\andinalog_iot_telemetry.csv
Tamaño: 28,920 filas y 10 columnas
0.13.2


### Interpretación sencilla

Cada fila debería representar una medición tomada durante un viaje. Leer inicialmente todas las columnas como texto evita que el programa corrija o transforme valores automáticamente antes de revisarlos.

## 2. Vista general del archivo

In [4]:
print("Nombres de las columnas:")
for numero, columna in enumerate(bronze.columns, start=1):
    print(f"{numero:>2}. {columna}")

print("\nPrimeras cinco filas:")
display(bronze.head())

Nombres de las columnas:
 1. timestamp
 2. viaje_id
 3. order_id
 4. camion_id
 5. producto_id
 6. temperatura_cabina_c
 7. temp_unit
 8. humedad_cabina_pct
 9. desviacion_termica_flag
10. desviacion_proximos_60min_flag

Primeras cinco filas:


,timestamp,viaje_id,order_id,camion_id,producto_id,temperatura_cabina_c,temp_unit,humedad_cabina_pct,desviacion_termica_flag,desviacion_proximos_60min_flag
0,2026-08-11 14:11:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,2.19,C,72.6,0,0
1,2026-08-11 14:41:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,4.91,C,75.0,0,0
2,2026-08-11 15:11:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,2.76,C,94.7,0,0
3,2026-08-11 15:41:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,3.96,C,74.0,0,0
4,2026-08-11 16:11:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,4.49,C,70.3,0,0


### ¿Qué significa cada columna?

| Columna | Explicación sencilla |
|---|---|
| `timestamp` | Fecha y hora de la lectura. |
| `viaje_id` | Identificador del viaje. |
| `order_id` | Orden relacionada con el viaje. |
| `camion_id` | Camión que transporta el producto. |
| `producto_id` | Producto transportado. |
| `temperatura_cabina_c` | Temperatura registrada dentro de la cabina. |
| `temp_unit` | Unidad utilizada para la temperatura. |
| `humedad_cabina_pct` | Porcentaje de humedad dentro de la cabina. |
| `desviacion_termica_flag` | Indica si existe una desviación térmica en ese momento. |
| `desviacion_proximos_60min_flag` | Indica si habrá una desviación durante la hora siguiente. |

El archivo contiene identificadores, una marca de tiempo, mediciones ambientales y dos indicadores de desviación térmica.

## 3. Valores vacíos

In [5]:
# Como el archivo se leyó como texto, un campo vacío aparece como una cadena vacía.
vacios = bronze.apply(lambda col: col.str.strip().eq("").sum())
resumen_vacios = pd.DataFrame({
    "campos_vacios": vacios,
    "porcentaje": (vacios / len(bronze) * 100).round(2),
}).sort_values("campos_vacios", ascending=False)

display(resumen_vacios)

columnas_con_vacios = resumen_vacios.query("campos_vacios > 0")
print(f"Columnas con al menos un campo vacío: {len(columnas_con_vacios)}")
print(f"Total de campos vacíos: {int(vacios.sum()):,}")

,campos_vacios,porcentaje
humedad_cabina_pct,100,0.35
temperatura_cabina_c,80,0.28
viaje_id,0,0.00
timestamp,0,0.00
camion_id,0,0.00
order_id,0,0.00
producto_id,0,0.00
temp_unit,0,0.00
desviacion_termica_flag,0,0.00
desviacion_proximos_60min_flag,0,0.00


Columnas con al menos un campo vacío: 2
Total de campos vacíos: 180


In [6]:
print("Interpretación sencilla")
if vacios.sum() == 0:
    print("No se encontraron campos vacíos.")
else:
    for columna, cantidad in vacios[vacios > 0].sort_values(ascending=False).items():
        porcentaje = cantidad / len(bronze) * 100
        print(f"- {columna}: {cantidad:,} valores vacíos ({porcentaje:.2f}% del archivo).")
    print("Estos valores todavía no se rellenan ni se eliminan; solamente quedan identificados para el tratamiento posterior.")

Interpretación sencilla
- humedad_cabina_pct: 100 valores vacíos (0.35% del archivo).
- temperatura_cabina_c: 80 valores vacíos (0.28% del archivo).
Estos valores todavía no se rellenan ni se eliminan; solamente quedan identificados para el tratamiento posterior.


## 4. Registros duplicados

In [7]:
duplicados_exactos = int(bronze.duplicated().sum())
duplicados_lectura = int(bronze.duplicated(["viaje_id", "timestamp"]).sum())

print("Filas completamente repetidas:", duplicados_exactos)
print("Combinaciones repetidas de viaje_id + timestamp:", duplicados_lectura)

if duplicados_lectura > 0:
    ejemplo_duplicados = (
        bronze.loc[bronze.duplicated(["viaje_id", "timestamp"], keep=False)]
        .sort_values(["viaje_id", "timestamp"])
        .head(10)
    )
    print("\nEjemplo de lecturas repetidas:")
    display(ejemplo_duplicados)

Filas completamente repetidas: 115
Combinaciones repetidas de viaje_id + timestamp: 120

Ejemplo de lecturas repetidas:


,timestamp,viaje_id,order_id,camion_id,producto_id,temperatura_cabina_c,temp_unit,humedad_cabina_pct,desviacion_termica_flag,desviacion_proximos_60min_flag
33,2026-08-13 04:00:00,VIA-00002,ORD-2026-04815,CAM-12,PROD-029,19.63,C,59.6,0,0
28832,2026-08-13 04:00:00,VIA-00002,ORD-2026-04815,CAM-12,PROD-029,19.63,C,59.6,0,0
647,2026-08-30 10:57:00,VIA-00027,ORD-2026-05466,CAM-20,PROD-026,3.31,C,78.1,0,0
28826,2026-08-30 10:57:00,VIA-00027,ORD-2026-05466,CAM-20,PROD-026,3.31,C,78.1,0,0
867,2026-08-01 01:56:00,VIA-00037,ORD-2026-01721,CAM-01,PROD-001,3.53,C,72.0,0,0
28910,2026-08-01 01:56:00,VIA-00037,ORD-2026-01721,CAM-01,PROD-001,3.53,C,72.0,0,0
959,2026-08-07 13:29:00,VIA-00040,ORD-2026-02804,CAM-20,PROD-060,-17.72,C,79.1,0,0
28908,2026-08-07 13:29:00,VIA-00040,ORD-2026-02804,CAM-20,PROD-060,-17.72,C,79.1,0,0
1109,2026-08-29 06:13:00,VIA-00047,ORD-2026-03263,CAM-20,PROD-028,21.22,C,50.4,0,0
28818,2026-08-29 06:13:00,VIA-00047,ORD-2026-03263,CAM-20,PROD-028,21.22,C,50.4,0,0


In [8]:
print("Interpretación sencilla")
print(f"Hay {duplicados_exactos:,} filas que se repiten exactamente.")
print(f"Además, hay {duplicados_lectura:,} repeticiones al usar viaje y momento como identificación de una lectura.")
if duplicados_lectura > duplicados_exactos:
    print("Esto significa que algunas lecturas comparten viaje y hora, aunque el contenido de sus demás columnas no es idéntico.")
print("En el siguiente notebook habrá que definir cuáles son copias y cuáles representan un conflicto que necesita revisión.")

Interpretación sencilla
Hay 115 filas que se repiten exactamente.
Además, hay 120 repeticiones al usar viaje y momento como identificación de una lectura.
Esto significa que algunas lecturas comparten viaje y hora, aunque el contenido de sus demás columnas no es idéntico.
En el siguiente notebook habrá que definir cuáles son copias y cuáles representan un conflicto que necesita revisión.


## 5. Revisión de identificadores

In [9]:
columnas_id = ["viaje_id", "order_id", "camion_id", "producto_id"]
resumen_id = pd.DataFrame({
    "cantidad_distinta": [bronze[col].nunique() for col in columnas_id],
    "campos_vacios": [bronze[col].str.strip().eq("").sum() for col in columnas_id],
}, index=columnas_id)
display(resumen_id)

patrones = {
    "viaje_id": r"VIA-\d{5}",
    "order_id": r"ORD-\d{4}-\d{5}",
    "camion_id": r"CAM-\d{2}",
    "producto_id": r"PROD-\d{3}",
}
fuera_patron = {
    col: int((~bronze[col].str.fullmatch(patron)).sum())
    for col, patron in patrones.items()
}
display(pd.Series(fuera_patron, name="valores_fuera_del_formato").to_frame())

,cantidad_distinta,campos_vacios
viaje_id,1200,0
order_id,1200,0
camion_id,48,0
producto_id,60,0


,valores_fuera_del_formato
viaje_id,0
order_id,0
camion_id,50
producto_id,0


In [10]:
print("Interpretación sencilla")
for columna in columnas_id:
    print(f"- {columna}: {bronze[columna].nunique():,} identificadores distintos y {fuera_patron[columna]:,} fuera del formato esperado.")
print("Este control revisa la forma escrita de los códigos; no demuestra por sí solo que cada código exista en los catálogos oficiales.")

Interpretación sencilla
- viaje_id: 1,200 identificadores distintos y 0 fuera del formato esperado.
- order_id: 1,200 identificadores distintos y 0 fuera del formato esperado.
- camion_id: 48 identificadores distintos y 50 fuera del formato esperado.
- producto_id: 60 identificadores distintos y 0 fuera del formato esperado.
Este control revisa la forma escrita de los códigos; no demuestra por sí solo que cada código exista en los catálogos oficiales.


## 6. Revisión de fechas y horas

In [11]:
# errors='coerce' se usa solo en una copia auxiliar del diagnóstico.
# El valor Bronze original permanece intacto.
timestamp_convertido = pd.to_datetime(bronze["timestamp"], errors="coerce")
fechas_invalidas = timestamp_convertido.isna()

print("Fechas y horas válidas:", int((~fechas_invalidas).sum()))
print("Fechas y horas no interpretables:", int(fechas_invalidas.sum()))

if fechas_invalidas.any():
    print("\nValores no interpretables más frecuentes:")
    display(bronze.loc[fechas_invalidas, "timestamp"].value_counts().head(10).to_frame("cantidad"))

print("\nPrimera fecha válida:", timestamp_convertido.min())
print("Última fecha válida:", timestamp_convertido.max())

Fechas y horas válidas: 28905
Fechas y horas no interpretables: 15

Valores no interpretables más frecuentes:


,cantidad
timestamp,
2026-02-31 09:15:00,15



Primera fecha válida: 2026-08-01 00:10:00
Última fecha válida: 2026-08-31 11:11:00


In [12]:
print("Interpretación sencilla")
if fechas_invalidas.any():
    print(f"Se encontraron {int(fechas_invalidas.sum()):,} fechas que el programa no puede reconocer como fechas reales.")
    print("Por ejemplo, un día 31 en un mes que no tiene 31 días es una fecha imposible.")
    print("No se corrigen aquí porque no conocemos con certeza cuál era la fecha verdadera.")
else:
    print("Todas las fechas pueden interpretarse correctamente.")

Interpretación sencilla
Se encontraron 15 fechas que el programa no puede reconocer como fechas reales.
Por ejemplo, un día 31 en un mes que no tiene 31 días es una fecha imposible.
No se corrigen aquí porque no conocemos con certeza cuál era la fecha verdadera.


## 7. Revisión de temperatura

In [13]:
temperatura_texto = bronze["temperatura_cabina_c"].str.strip()
temperatura_num = pd.to_numeric(temperatura_texto.replace("", np.nan), errors="coerce")
no_numericas_temp = temperatura_texto.ne("") & temperatura_num.isna()
centinela_999 = temperatura_num.eq(-999)

print("Unidades encontradas:")
display(bronze["temp_unit"].value_counts(dropna=False).rename("cantidad").to_frame())
print("Temperaturas vacías:", int(temperatura_texto.eq("").sum()))
print("Temperaturas no numéricas:", int(no_numericas_temp.sum()))
print("Valores centinela -999:", int(centinela_999.sum()))

temperaturas_reales = temperatura_num.loc[temperatura_num.notna() & ~centinela_999]
display(temperaturas_reales.describe().round(2).to_frame("temperatura_registrada"))

Unidades encontradas:


,cantidad
temp_unit,
C,28865
F,50
K,5


Temperaturas vacías: 80
Temperaturas no numéricas: 0
Valores centinela -999: 120


,temperatura_registrada
count,28720.00
mean,8.44
std,14.61
min,-21.63
25%,3.44
50%,15.61
75%,20.05
max,291.22


In [14]:
print("Interpretación sencilla")
conteo_unidades = bronze["temp_unit"].value_counts()
for unidad, cantidad in conteo_unidades.items():
    print(f"- Unidad {unidad}: {cantidad:,} lecturas.")
print(f"También existen {int(temperatura_texto.eq('').sum()):,} temperaturas vacías y {int(centinela_999.sum()):,} valores -999.")
print("El valor -999 es una marca de ausencia o error, no una temperatura real.")
print("Las lecturas en Fahrenheit pueden convertirse de forma exacta a Celsius en el próximo notebook.")
print("Las lecturas en Kelvin requieren revisión porque esa unidad no se espera en la operación definida.")

Interpretación sencilla
- Unidad C: 28,865 lecturas.
- Unidad F: 50 lecturas.
- Unidad K: 5 lecturas.
También existen 80 temperaturas vacías y 120 valores -999.
El valor -999 es una marca de ausencia o error, no una temperatura real.
Las lecturas en Fahrenheit pueden convertirse de forma exacta a Celsius en el próximo notebook.
Las lecturas en Kelvin requieren revisión porque esa unidad no se espera en la operación definida.


## 8. Revisión de humedad

In [15]:
humedad_texto = bronze["humedad_cabina_pct"].str.strip()
humedad_num = pd.to_numeric(humedad_texto.replace("", np.nan), errors="coerce")
no_numericas_humedad = humedad_texto.ne("") & humedad_num.isna()
fuera_rango_humedad = humedad_num.notna() & ~humedad_num.between(0, 100)

print("Humedades vacías:", int(humedad_texto.eq("").sum()))
print("Humedades no numéricas:", int(no_numericas_humedad.sum()))
print("Humedades fuera del rango 0% a 100%:", int(fuera_rango_humedad.sum()))
display(humedad_num.describe().round(2).to_frame("humedad_pct"))

Humedades vacías: 100
Humedades no numéricas: 0
Humedades fuera del rango 0% a 100%: 15


,humedad_pct
count,28820.00
mean,61.41
std,15.70
min,-92.60
25%,48.00
50%,60.30
75%,75.10
max,100.00


In [16]:
print("Interpretación sencilla")
print(f"Hay {int(humedad_texto.eq('').sum()):,} campos de humedad vacíos.")
print(f"Hay {int(fuera_rango_humedad.sum()):,} valores fuera del rango físicamente posible de 0% a 100%.")
print("En este diagnóstico solo los identificamos; el próximo notebook decidirá su tratamiento sin inventar mediciones.")

Interpretación sencilla
Hay 100 campos de humedad vacíos.
Hay 15 valores fuera del rango físicamente posible de 0% a 100%.
En este diagnóstico solo los identificamos; el próximo notebook decidirá su tratamiento sin inventar mediciones.


## 9. Revisión de indicadores de desviación

In [17]:
columnas_flag = ["desviacion_termica_flag", "desviacion_proximos_60min_flag"]
for columna in columnas_flag:
    print(f"\n{columna}")
    conteo = bronze[columna].value_counts(dropna=False).sort_index()
    resumen = pd.DataFrame({
        "cantidad": conteo,
        "porcentaje": (conteo / len(bronze) * 100).round(2),
    })
    display(resumen)


desviacion_termica_flag


,cantidad,porcentaje
desviacion_termica_flag,,
0,27969,96.71
1,951,3.29



desviacion_proximos_60min_flag


,cantidad,porcentaje
desviacion_proximos_60min_flag,,
0,27578,95.36
1,1342,4.64


In [18]:
print("Interpretación sencilla")
actuales = int((bronze["desviacion_termica_flag"] == "1").sum())
futuras = int((bronze["desviacion_proximos_60min_flag"] == "1").sum())
print(f"- {actuales:,} lecturas ({actuales / len(bronze) * 100:.2f}%) indican una desviación en ese momento.")
print(f"- {futuras:,} lecturas ({futuras / len(bronze) * 100:.2f}%) indican una desviación durante la hora siguiente.")
print("El indicador actual y el futuro no significan lo mismo: uno describe el presente y el otro la ventana posterior de 60 minutos.")

Interpretación sencilla
- 951 lecturas (3.29%) indican una desviación en ese momento.
- 1,342 lecturas (4.64%) indican una desviación durante la hora siguiente.
El indicador actual y el futuro no significan lo mismo: uno describe el presente y el otro la ventana posterior de 60 minutos.


## 10. Consistencia dentro de cada viaje

In [19]:
consistencia = bronze.groupby("viaje_id").agg(
    ordenes=("order_id", "nunique"),
    camiones=("camion_id", "nunique"),
    productos=("producto_id", "nunique"),
    lecturas=("timestamp", "size"),
)

resumen_consistencia = pd.DataFrame({
    "viajes_con_mas_de_una_orden": [(consistencia["ordenes"] > 1).sum()],
    "viajes_con_mas_de_un_camion": [(consistencia["camiones"] > 1).sum()],
    "viajes_con_mas_de_un_producto": [(consistencia["productos"] > 1).sum()],
})
display(resumen_consistencia)
display(consistencia["lecturas"].describe().round(2).to_frame("lecturas_por_viaje"))

,viajes_con_mas_de_una_orden,viajes_con_mas_de_un_camion,viajes_con_mas_de_un_producto
0,0,50,0


,lecturas_por_viaje
count,1200.00
mean,24.10
std,0.32
min,24.00
25%,24.00
50%,24.00
75%,24.00
max,26.00


In [20]:
print("Interpretación sencilla")
print("Este control revisa si un mismo viaje cambia de orden, camión o producto dentro del archivo.")
print("Un cambio no siempre es un error, pero debe revisarse porque puede contradecir el significado esperado de un viaje.")

Interpretación sencilla
Este control revisa si un mismo viaje cambia de orden, camión o producto dentro del archivo.
Un cambio no siempre es un error, pero debe revisarse porque puede contradecir el significado esperado de un viaje.


## 11. Resumen del diagnóstico

In [21]:
resumen_final = pd.DataFrame({
    "revisión": [
        "Filas", "Columnas", "Campos vacíos", "Duplicados exactos",
        "Duplicados por viaje + timestamp", "Fechas no interpretables",
        "Temperaturas -999", "Unidades Fahrenheit", "Unidades Kelvin",
        "Humedades fuera de 0-100", "Valores no binarios en los indicadores",
    ],
    "resultado": [
        len(bronze), bronze.shape[1], int(vacios.sum()), duplicados_exactos,
        duplicados_lectura, int(fechas_invalidas.sum()), int(centinela_999.sum()),
        int((bronze["temp_unit"] == "F").sum()), int((bronze["temp_unit"] == "K").sum()),
        int(fuera_rango_humedad.sum()),
        int(sum((~bronze[col].isin(["0", "1"])).sum() for col in columnas_flag)),
    ],
})
display(resumen_final)

,revisión,resultado
0,Filas,28920
1,Columnas,10
2,Campos vacíos,180
3,Duplicados exactos,115
4,Duplicados por viaje + timestamp,120
5,Fechas no interpretables,15
6,Temperaturas -999,120
7,Unidades Fahrenheit,50
8,Unidades Kelvin,5
9,Humedades fuera de 0-100,15


### Conclusión explicada de forma sencilla

El archivo contiene la información necesaria para estudiar el comportamiento térmico de los viajes, pero **todavía no está listo para analizarse o entrenar un modelo**.

Los principales puntos que deberá resolver el siguiente notebook son:

1. Distinguir y tratar los registros duplicados.
2. Conservar como error las fechas imposibles cuando no exista una corrección segura.
3. Convertir Fahrenheit a Celsius y revisar las lecturas en Kelvin.
4. Reconocer `-999` como una marca de ausencia, no como una temperatura real.
5. Tratar los campos vacíos sin inventar mediciones.
6. Mantener separada la desviación actual de la desviación esperada durante la hora siguiente.

Ninguna fila fue eliminada y ningún dato original fue modificado durante este diagnóstico.